# IMDb Movies 2016–2025: Data Collection

This notebook collects a ** movie dataset for 2016–2025** from IMDb's official non-commercial bulk datasets using **HTTP requests**.

**Business goal:** build a clean, reproducible starting dataset for analysing movie volume, genres, ratings, popularity and runtime.

We will:

1. Send HTTP requests to IMDb's official dataset files and check the status code.
2. Read only the columns needed for this project, in **chunks**, so the multi-million-row IMDb files never sit fully in memory.
3. Keep only titles of type `movie` released from 2016 to 2025.
4. Use a **US + English title record** as a simple Hollywood-focused proxy.
5. Combine movie details with IMDb ratings and votes.
6. Save the extracted dataset as a CSV for the cleaning notebook.

## 1) Import the Required Libraries

We keep the imports small: `requests` for the HTTP calls, `pandas` for chunked reading and merging, and `os` for creating folders.

In [1]:
import requests
import pandas as pd
import os

## 2) Set the IMDb Dataset URLs

IMDb provides the datasets as compressed TSV files. We use only the files required for this project.

In [2]:
BASIC_URL = "https://datasets.imdbws.com/title.basics.tsv.gz"
AKAS_URL = "https://datasets.imdbws.com/title.akas.tsv.gz"
RATINGS_URL = "https://datasets.imdbws.com/title.ratings.tsv.gz"

start_year = 2016
end_year = 2025

raw_folder = "../data/raw"
os.makedirs(raw_folder, exist_ok=True)

## 3) Check the HTTP Requests

Before processing, we confirm that each official file can be reached. `response.status_code` should be `200`, and `raise_for_status()` stops the notebook with a clear error if a request fails, so a broken download never turns into silently empty data.

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

for name, url in {
    "title.basics": BASIC_URL,
    "title.akas": AKAS_URL,
    "title.ratings": RATINGS_URL
}.items():
    response = requests.get(url, headers=headers, stream=True, timeout=30)
    print(f"{name:<15} -> {response.status_code}")
    response.raise_for_status()

title.basics    -> 200
title.akas      -> 200
title.ratings   -> 200


## 4) Extract Movies from 2016–2025

`title.basics` contains many different title types. We keep only:

- `titleType = movie`
- `startYear` from 2016 to 2025

The file is processed in chunks so the full IMDb title dataset is not loaded into memory at once.

In [4]:
basic_parts = []

for chunk in pd.read_csv(
    BASIC_URL,
    sep="\t",
    na_values="\\N",
    dtype=str,
    usecols=[
        "tconst",
        "titleType",
        "primaryTitle",
        "originalTitle",
        "startYear",
        "runtimeMinutes",
        "genres"
    ],
    chunksize=100000
):
    chunk = chunk[chunk["titleType"] == "movie"]

    chunk["startYear"] = pd.to_numeric(
        chunk["startYear"],
        errors="coerce"
    )

    chunk = chunk[
        chunk["startYear"].between(start_year, end_year)
    ]

    basic_parts.append(chunk)

movies = pd.concat(
    basic_parts,
    ignore_index=True
)

print(movies.shape)
movies.head()

(199036, 7)


,tconst,titleType,primaryTitle,originalTitle,startYear,runtimeMinutes,genres
0,tt0011801,movie,Tötet nicht mehr,Tötet nicht mehr,2019.0,NaN,"Action,Crime"
1,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,2020.0,70,"Drama,Mystery"
2,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,2022.0,85,Documentary
3,tt0069049,movie,The Other Side of the Wind,The Other Side of the Wind,2018.0,122,Drama
4,tt0070596,movie,Socialist Realism,El realismo socialista,2023.0,78,"Drama,History"


## 5) Keep US + English Title Records

**Known limitation:** a US + English alias only shows that a title has a US English-language listing. It does not prove the film is a Hollywood production, so some international films (for example, Indian-language titles with an English US alias) can pass this filter. A stricter definition could use production country or original language, which are not in the files used here. The EDA notebook quantifies the effect of this choice where it matters.

In [5]:
aka_parts = []

for chunk in pd.read_csv(
    AKAS_URL,
    sep="\t",
    na_values="\\N",
    dtype=str,
    usecols=["titleId", "region", "language"],
    chunksize=100000
):
    chunk = chunk[
        (chunk["region"] == "US") &
        (chunk["language"] == "en")
    ]

    aka_parts.append(chunk)

us_english = pd.concat(
    aka_parts,
    ignore_index=True
)

us_english = us_english.drop_duplicates(
    subset="titleId"
)

print(us_english.shape)
us_english.head()

(4455, 3)


,titleId,region,language
0,tt0000969,US,en
1,tt0002461,US,en
2,tt0005809,US,en
3,tt0008305,US,en
4,tt0009968,US,en


## 6) Merge the Hollywood-focused Filter

We keep only the movies that have a US + English title record (an **inner merge**, so movies without such a record are dropped). The first cell makes the step safe to re-run in the same notebook session.

In [6]:
# Make this cell safe to rerun in an existing notebook session.
movies = movies.drop(
    columns=[c for c in ["titleId", "region", "language"] if c in movies.columns]
)

movies = movies.merge(
    us_english,
    left_on="tconst",
    right_on="titleId",
    how="inner"
)

movies = movies.drop(
    columns=["titleId"]
)

print(movies.shape)
movies.head()

(648, 9)


,tconst,titleType,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,region,language
0,tt0437086,movie,Alita: Battle Angel,Alita: Battle Angel,2019.0,122,"Action,Adventure,Sci-Fi",US,en
1,tt0800325,movie,The Dirt,The Dirt,2019.0,107,"Biography,Comedy,Drama",US,en
2,tt0837563,movie,Pet Sematary,Pet Sematary,2019.0,100,"Horror,Mystery,Thriller",US,en
3,tt10017680,movie,P Storm,'P' fung bou,2019.0,96,"Action,Crime",US,en
4,tt10055770,movie,Vellai Pookal,Vellaipookal,2019.0,122,Thriller,US,en


## 7) Collect IMDb Ratings and Votes

We read the ratings dataset and keep only the movie IDs already present in our filtered movie list.

In [7]:
movie_ids = set(movies["tconst"])

rating_parts = []

for chunk in pd.read_csv(
    RATINGS_URL,
    sep="\t",
    dtype={
        "tconst": str,
        "averageRating": float,
        "numVotes": int
    },
    chunksize=100000
):
    chunk = chunk[
        chunk["tconst"].isin(movie_ids)
    ]

    rating_parts.append(chunk)

ratings = pd.concat(
    rating_parts,
    ignore_index=True
)

ratings.head()

,tconst,averageRating,numVotes
0,tt0437086,7.3,328572
1,tt0800325,7.0,61733
2,tt0837563,5.7,107421
3,tt10017680,5.9,892
4,tt10055770,7.1,2038


## 8) Merge Movie Details with Ratings

The final raw dataset contains movie year, runtime, genres, IMDb rating and vote count.

This is a **left merge**: every movie that passed the filter is kept, even if it has no rating yet. Such movies get missing (`NaN`) rating and vote values, which are handled explicitly in the cleaning and EDA notebooks rather than being dropped here.

In [8]:
final_df = movies.merge(
    ratings,
    on="tconst",
    how="left"
)

final_df = final_df[
    [
        "tconst",
        "primaryTitle",
        "originalTitle",
        "startYear",
        "runtimeMinutes",
        "genres",
        "region",
        "language",
        "averageRating",
        "numVotes"
    ]
]

print(final_df.shape)
final_df.head()

(648, 10)


,tconst,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,region,language,averageRating,numVotes
0,tt0437086,Alita: Battle Angel,Alita: Battle Angel,2019.0,122,"Action,Adventure,Sci-Fi",US,en,7.3,328572.0
1,tt0800325,The Dirt,The Dirt,2019.0,107,"Biography,Comedy,Drama",US,en,7.0,61733.0
2,tt0837563,Pet Sematary,Pet Sematary,2019.0,100,"Horror,Mystery,Thriller",US,en,5.7,107421.0
3,tt10017680,P Storm,'P' fung bou,2019.0,96,"Action,Crime",US,en,5.9,892.0
4,tt10055770,Vellai Pookal,Vellaipookal,2019.0,122,Thriller,US,en,7.1,2038.0


## 9) Inspect the Extracted Data

In [9]:
final_df.head()

,tconst,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,region,language,averageRating,numVotes
0,tt0437086,Alita: Battle Angel,Alita: Battle Angel,2019.0,122,"Action,Adventure,Sci-Fi",US,en,7.3,328572.0
1,tt0800325,The Dirt,The Dirt,2019.0,107,"Biography,Comedy,Drama",US,en,7.0,61733.0
2,tt0837563,Pet Sematary,Pet Sematary,2019.0,100,"Horror,Mystery,Thriller",US,en,5.7,107421.0
3,tt10017680,P Storm,'P' fung bou,2019.0,96,"Action,Crime",US,en,5.9,892.0
4,tt10055770,Vellai Pookal,Vellaipookal,2019.0,122,Thriller,US,en,7.1,2038.0


In [10]:
final_df.shape

(648, 10)

In [11]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 648 entries, 0 to 647
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tconst          648 non-null    str    
 1   primaryTitle    648 non-null    str    
 2   originalTitle   648 non-null    str    
 3   startYear       648 non-null    float64
 4   runtimeMinutes  631 non-null    str    
 5   genres          646 non-null    str    
 6   region          648 non-null    str    
 7   language        648 non-null    str    
 8   averageRating   619 non-null    float64
 9   numVotes        619 non-null    float64
dtypes: float64(3), str(7)
memory usage: 50.8 KB


## 10) Save the Raw Dataset

This CSV is the input for `02_imdb_data_cleaning.ipynb`. No cleaning has been applied yet, so the raw file can always be regenerated and compared with the cleaned one.

In [12]:
raw_file_path = "../data/raw/imdb_hollywood_movies_2016_2025_raw.csv"

final_df.to_csv(
    raw_file_path,
    index=False
)

print("Saved:", raw_file_path)

Saved: ../data/raw/imdb_hollywood_movies_2016_2025_raw.csv
